In [ ]:

#Week 4: All Anomaly Detection Models (Independent Version)


import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import SGDOneClassSVM
from sklearn.pipeline import make_pipeline
from sklearn.cluster import DBSCAN
from sklearn.neighbors import KNeighborsClassifier
import tensorflow as tf
from tensorflow import keras

FEATURE_COLS = [
    'login_count', 'device_event_count', 'http_event_count',
    'unique_pc_count', 'after_hours_count', 'is_weekend',
    'first_login_hour', 'last_activity_hour'
]

def get_scaled(feature_vectors):
    X = feature_vectors[FEATURE_COLS].fillna(0)
    scaler = StandardScaler()
    return scaler.fit_transform(X)


# ── Model 1: Isolation Forest ─────────────────────────────────────────────────

def run_isolation_forest(feature_vectors):
    print("\n--- Model 1: Isolation Forest ---")
    X_scaled = get_scaled(feature_vectors)

    model = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
    preds = model.fit_predict(X_scaled)

    result = feature_vectors[['user', 'day']].copy()
    result['if_anomaly'] = (preds == -1).astype(int)

    n = result['if_anomaly'].sum()
    print(f"Anomalous user-days: {n} ({n/len(result)*100:.1f}%)")
    return result


# ── Model 2: One-Class SVM ────────────────────────────────────────────────────

def run_one_class_svm(feature_vectors):
    print("\n--- Model 2: One-Class SVM ---")
    X_scaled = get_scaled(feature_vectors)

    svm_pipeline = make_pipeline(
        Nystroem(kernel='rbf', gamma=None, n_components=500, random_state=42),
        SGDOneClassSVM(nu=0.05, random_state=42)
    )
    svm_pipeline.fit(X_scaled)
    preds = svm_pipeline.predict(X_scaled)

    result = feature_vectors[['user', 'day']].copy()
    result['svm_anomaly'] = (preds == -1).astype(int)

    n = result['svm_anomaly'].sum()
    print(f"Anomalous user-days: {n} ({n/len(result)*100:.1f}%)")
    return result


# ── Model 3: Autoencoder ──────────────────────────────────────────────────────

def run_autoencoder(feature_vectors):
    print("\n--- Model 3: Autoencoder ---")
    X_scaled = get_scaled(feature_vectors)
    input_dim = X_scaled.shape[1]

    input_layer = keras.Input(shape=(input_dim,))
    encoded = keras.layers.Dense(8, activation='relu')(input_layer)
    encoded = keras.layers.Dense(4, activation='relu')(encoded)
    decoded = keras.layers.Dense(8, activation='relu')(encoded)
    decoded = keras.layers.Dense(input_dim, activation='linear')(decoded)

    autoencoder = keras.Model(input_layer, decoded)
    autoencoder.compile(optimizer='adam', loss='mse')
    autoencoder.fit(X_scaled, X_scaled, epochs=20, batch_size=256,
                    validation_split=0.1, verbose=0)

    reconstructed = autoencoder.predict(X_scaled, verbose=0)
    mse = np.mean(np.power(X_scaled - reconstructed, 2), axis=1)
    threshold = np.percentile(mse, 95)

    result = feature_vectors[['user', 'day']].copy()
    result['ae_anomaly'] = (mse > threshold).astype(int)
    result['ae_reconstruction_error'] = mse

    n = result['ae_anomaly'].sum()
    print(f"Threshold: {threshold:.4f}")
    print(f"Anomalous user-days: {n} ({n/len(result)*100:.1f}%)")
    return result


# ── Model 4: DBSCAN ───────────────────────────────────────────────────────────

def run_dbscan(feature_vectors):
    print("\n--- Model 4: DBSCAN ---")
    X_scaled = get_scaled(feature_vectors)

    pca = PCA(n_components=5, random_state=42)
    X_reduced = pca.fit_transform(X_scaled)
    print(f"PCA variance retained: {pca.explained_variance_ratio_.sum():.3f}")

    # Stratified sample - 50k rows, every user represented
    fv_reset = feature_vectors.reset_index(drop=True)
    sample_idx = (
        fv_reset.groupby('user', group_keys=False)
        .apply(lambda g: g.sample(min(len(g), 15), random_state=42))
        .index
    )
    remaining = list(set(range(len(fv_reset))) - set(sample_idx))
    extra_needed = max(0, 50000 - len(sample_idx))
    if extra_needed > 0:
        extra_idx = np.random.choice(remaining, size=extra_needed, replace=False)
        sample_idx = list(sample_idx) + list(extra_idx)

    X_sample = X_reduced[sample_idx]
    model = DBSCAN(eps=0.8, min_samples=5, algorithm='ball_tree', n_jobs=-1)
    sample_labels = model.fit_predict(X_sample)

    n_clusters = len(set(sample_labels)) - (1 if -1 in sample_labels else 0)
    print(f"Clusters found: {n_clusters}")

    # Extend to full dataset via KNN
    knn = KNeighborsClassifier(n_neighbors=3, algorithm='ball_tree', n_jobs=-1)
    knn.fit(X_sample, sample_labels)
    all_labels = knn.predict(X_reduced)

    result = feature_vectors[['user', 'day']].copy()
    result['dbscan_anomaly'] = (all_labels == -1).astype(int)
    result['dbscan_cluster'] = all_labels

    n = result['dbscan_anomaly'].sum()
    print(f"Anomalous user-days: {n} ({n/len(result)*100:.1f}%)")
    return result


# ── Alert Generation ──────────────────────────────────────────────────────────

def generate_alerts(feature_vectors, if_result, svm_result, ae_result, dbscan_result):
    print("\n--- Alert Generation ---")

    # Merge all model results on user + day
    merged = feature_vectors.copy()
    merged = merged.merge(if_result[['user', 'day', 'if_anomaly']], on=['user', 'day'], how='left')
    merged = merged.merge(svm_result[['user', 'day', 'svm_anomaly']], on=['user', 'day'], how='left')
    merged = merged.merge(ae_result[['user', 'day', 'ae_anomaly', 'ae_reconstruction_error']], on=['user', 'day'], how='left')
    merged = merged.merge(dbscan_result[['user', 'day', 'dbscan_anomaly']], on=['user', 'day'], how='left')

    anomaly_cols = ['if_anomaly', 'svm_anomaly', 'ae_anomaly', 'dbscan_anomaly']
    merged['model_flag_count'] = merged[anomaly_cols].sum(axis=1)

    print("Flag count distribution:")
    print(merged['model_flag_count'].value_counts().sort_index())

    # Alert if 2+ models agree
    alerts = merged[merged['model_flag_count'] >= 2].copy()
    alerts['models_flagged'] = alerts[anomaly_cols].apply(
        lambda row: [col.replace('_anomaly', '') for col in anomaly_cols if row[col] == 1],
        axis=1
    )

    alerts_summary = alerts[[
        'user', 'day', 'model_flag_count', 'models_flagged',
        'login_count', 'http_event_count', 'device_event_count', 'after_hours_count'
    ]].sort_values('model_flag_count', ascending=False).reset_index(drop=True)

    print(f"\nTotal alerts (>=2 models): {len(alerts_summary)}")
    print("\nTop 20 alerts:")
    print(alerts_summary.head(20))

    user_alerts = (
        alerts_summary.groupby('user')
        .agg(total_alert_days=('day', 'count'), max_models_flagged=('model_flag_count', 'max'))
        .sort_values('total_alert_days', ascending=False)
        .reset_index()
    )
    print("\nMost alerted users:")
    print(user_alerts.head(15))

    high_conf = alerts_summary[alerts_summary['model_flag_count'] == 4]
    print(f"\nFlagged by all 4 models: {len(high_conf)}")
    if len(high_conf) > 0:
        print(high_conf[['user', 'day', 'login_count', 'http_event_count']].head(10))

    return alerts_summary, user_alerts,merged



if_result   = run_isolation_forest(feature_vectors)
svm_result  = run_one_class_svm(feature_vectors)
ae_result   = run_autoencoder(feature_vectors)
dbscan_result = run_dbscan(feature_vectors)

alerts_summary, user_alerts, merged = generate_alerts(
    feature_vectors, if_result, svm_result, ae_result, dbscan_result
)

In [ ]:
# Export for Splunk 


export_full = merged.copy()
export_full['anomaly_score'] = export_full['model_flag_count'] / 4  # normalize 0-1
export_full['risk_label'] = np.where(
    export_full['model_flag_count'] >= 2, 'anomalous', 'normal'
)
export_full = export_full.rename(columns={'user': 'user_id', 'day': 'date'})
export_full.to_csv('insider_threat_scores.csv', index=False)
print(f"\nSaved insider_threat_scores.csv: {export_full.shape}")


alerts_export = alerts_summary.rename(columns={'user': 'user_id', 'day': 'date'})
alerts_summary['models_flagged'] = alerts_summary['models_flagged'].apply(lambda x: ','.join(x))
alerts_export.to_csv('insider_threat_alerts.csv', index=False)
print(f"Saved insider_threat_alerts.csv: {alerts_export.shape}")


user_alerts_export = user_alerts.rename(columns={'user': 'user_id'})
user_alerts_export.to_csv('insider_threat_user_summary.csv', index=False)
print(f"Saved insider_threat_user_summary.csv: {user_alerts_export.shape}")